# Table 1 — Selected OmegAMP leads (with sequences)

Reproduces Table 1 from `data/{mic,hc50,cc50}.csv` + the reference table (prototypes, leads, antibiotic controls). MIC50 = median over the 20-strain panel; SW = HC50/MIC50. Saves `../figures/table1.csv`.


In [1]:
# Table 1 -- Selected OmegAMP leads across generation modes (with sequences).
# Reproduced from data/{mic,hc50,cc50}.csv + the reference table. Saves ../figures/table1.csv.
# Conventions: MIC50 = median over the 20-strain panel (censored ">64" kept at its bound;
# if the median order-statistic is censored the result is reported as ">64"). SW = HC50/MIC50,
# with ">" propagated from any censored input. CC50/HC50 rounded; values above 128 shown as ">128".
import csv
from pathlib import Path
import pandas as pd

DATA = Path("../data")
rows = lambda f: list(csv.DictReader(open(DATA / f)))
mic = {r["short_name"]: r for r in rows("mic.csv")}
hc  = {r["short_name"]: r["HC50"] for r in rows("hc50.csv")}
cc  = {r["short_name"]: r["CC50"] for r in rows("cc50.csv")}
ref = {r["short_name"]: r for r in rows("omegamp_reference_table.csv")}
STRAINS = [c for c in next(iter(mic.values())) if c != "short_name"]
NAMED = {"Ab CRAB": "BAA-1605", "Kp ESBL": "BAA-2342", "Pa FQR": "BAA-3197", "Ef VRE": "700221"}
MDR_TOK = ["BAA-1605", "AIC222", "BAA-3170", "BAA-2342", "BAA-3197", "BAA-1556", "700802", "700221"]
MDR = [s for s in STRAINS if any(t in s for t in MDR_TOK)]
col = lambda tok: next(s for s in STRAINS if tok in s)

def pv(x):
    """Parse an assay cell -> (bound, censored?). '>64'->(64,True); '<2'->(2,True); '14.3'->(14.3,False); ''->(None,False)."""
    x = (x or "").strip()
    x2 = x[1:].strip() if x[:1] in "<>" else x
    try: return float(x2), (x[:1] in "<>")
    except ValueError: return None, False

def g(v): return str(int(v)) if float(v).is_integer() else f"{v:g}"

def mode_of(sn):
    """Mode as printed in Table 1 = the short-name prefix of Omega-<MODE>-...
    (not generation_mode: the reference table stores de-novo modes as
    "OmegAMP-P"/"-T"/"-U", which drops the leading D)."""
    return sn.split("-")[1] if sn.startswith("\u03a9-") else ""

def cc_show(x):
    n, c = pv(x)
    return "" if n is None else (f">{round(n)}" if c else str(round(n)))

def mic50(sn):
    """(bound, censored) censored-aware median across the panel."""
    r = mic.get(sn)
    if not r: return None, False
    ps = sorted((pv(r[s]) for s in STRAINS if pv(r[s])[0] is not None), key=lambda t: t[0])
    n = len(ps)
    if n == 0: return None, False
    if n % 2: return ps[n // 2]
    a, b = ps[n // 2 - 1], ps[n // 2]
    return (b[0], True) if b[1] else ((a[0] + b[0]) / 2, False)

def mic50_show(m):
    v, c = m
    return "" if v is None else (f">{g(v)}" if c else g(v))

def count_le2(sn, strain_set):
    r = mic.get(sn)
    if not r: return None
    n = sum(1 for s in strain_set if (pv(r[s])[0] is not None and not pv(r[s])[1] and pv(r[s])[0] <= 2))
    return f"{n}/{len(strain_set)}"

def sw(sn, m):
    h, hc_c = pv(hc.get(sn)); v, v_c = m
    if h is None or v in (None, 0): return ""
    q = round(h / v)
    return f">{q}" if (hc_c or v_c) else f"{q}"

SECTIONS = [
    ("Antibiotic controls", ["Polymyxin B", "Levofloxacin"]),
    ("De novo generation", ["Ω-DP-52", "Ω-DP-19"]),
    ("Inactive-to-active conversion",
     ["BoCo1", "Ω-AT-BoCo1-5", "Ω-AT-BoCo1-9", "GQ20", "Ω-AU-GQ20-4", "Mammutin-1", "Ω-AP-Mammutin-1-4"]),
    ("LPS binding",
     ["cecropin", "Ω-AMT-cecropin-1", "Ω-AMT-cecropin-4", "pa4", "Ω-AMT-pa4-1", "sarcotoxin", "Ω-AMT-sarcotoxin-4"]),
    ("DNA binding", ["bZIP", "Ω-MT-bZIP-8"]),
]

out = []
for section, names in SECTIONS:
    out.append({"Peptide": section})
    for sn in names:
        r = mic.get(sn, {}); m = mic50(sn)
        out.append({
            "Peptide": sn, "Sequence": ref.get(sn, {}).get("sequence", ""),
            "Mode": mode_of(sn),
            "Ab CRAB": r.get(col(NAMED["Ab CRAB"]), ""), "Kp ESBL": r.get(col(NAMED["Kp ESBL"]), ""),
            "Pa FQR": r.get(col(NAMED["Pa FQR"]), ""),   "Ef VRE": r.get(col(NAMED["Ef VRE"]), ""),
            "MIC50": mic50_show(m), "Strains<=2": count_le2(sn, STRAINS), "MDR<=2": count_le2(sn, MDR),
            "CC50": cc_show(cc.get(sn)), "SW": sw(sn, m),
        })
COLS = ["Peptide", "Sequence", "Mode", "Ab CRAB", "Kp ESBL", "Pa FQR", "Ef VRE",
        "MIC50", "Strains<=2", "MDR<=2", "CC50", "SW"]
df = pd.DataFrame(out).reindex(columns=COLS).fillna("")
import os; os.makedirs("../figures", exist_ok=True)
df.to_csv("../figures/table1.csv", index=False)
print("saved ../figures/table1.csv")
with pd.option_context("display.max_colwidth", 42, "display.width", 200):
    print(df.to_string(index=False))

saved ../figures/table1.csv
                      Peptide                                Sequence Mode Ab CRAB Kp ESBL Pa FQR Ef VRE MIC50 Strains<=2 MDR<=2 CC50   SW
          Antibiotic controls                                                                                                             
                  Polymyxin B                                                0.125   0.125    0.5    >64 0.375      15/20    5/8          
                 Levofloxacin                                                    8     >64    >64      4 0.375      12/20    2/8          
           De novo generation                                                                                                             
                      Ω-DP-52                       KLWKLAKKALKALGKVL   DP     0.5       1      1      2     1      18/20    7/8    2 >128
                      Ω-DP-19   KWKLFKKIEKVGRNIRDGIIKAGPAVAVVGQAASLAK   DP     0.3     0.5    0.5    >64   0.5      15/20    5/8   20 >256